In [ ]:
import os
import random
from PIL import Image, ImageOps, ImageFilter
import pytesseract
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from difflib import SequenceMatcher
import csv

# 📁 Folder containing .png and .gt.txt
data_dir = "dataset"
max_samples = 70  # 🔢 Limit to 20 samples

def preprocess_image(img_path):
    # monochrome filter
    img = Image.open(img_path).convert("L")
    
    # resize cuz tesseract better with bigger image
    scale = 2  # 2x upscale
    img = img.resize((img.width * scale, img.height * scale), Image.LANCZOS)
    
    # anti-aliasing (mirip2 lah)
    img = img.filter(ImageFilter.MedianFilter())

    return img

# 💡 Utility functions
def normalized_similarity(pred, truth):
    """String similarity between OCR result and label."""
    return SequenceMatcher(None, pred.strip(), truth.strip()).ratio()

def binarize(pred, truth, threshold=0.8):
    """1 if prediction is similar enough to ground truth."""
    return 1 if normalized_similarity(pred, truth) >= threshold else 0

valid_exts = (".png", ".jpg", ".jpeg")

# everyday i'm shufflin'
all_images = [f for f in os.listdir(data_dir) if f.endswith(valid_exts)]
random.shuffle(all_images)
selected_images = all_images[:max_samples]

# Evaluate array
y_true, y_pred, binary_eval = [], [], []

for fname in selected_images:
    base = fname[:-4]
    img_path = os.path.join(data_dir, fname)
    label_path = os.path.join(data_dir, base + ".gt.txt")

    if not os.path.exists(label_path):
        continue

    # OCR
    preprocessed_img = preprocess_image(img_path)
    ocr_text = pytesseract.image_to_string(preprocessed_img).strip()

    # Ground truth
    with open(label_path, "r", encoding="utf-8") as f:
        true_text = f.read().strip()

    y_pred.append(ocr_text)
    y_true.append(true_text)
    binary_eval.append(binarize(ocr_text, true_text))

# Metrics
accuracy = accuracy_score([1]*len(binary_eval), binary_eval)
precision, recall, f1, _ = precision_recall_fscore_support(
    [1]*len(binary_eval), binary_eval, average="binary", zero_division=0
)

# Print the metrics
print("Evaluation on 100 Samples:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# 🧾 Optional: Export results to CSV
csv_path = "tesseract_iiit5k_sample_results.csv"
with open(csv_path, "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Filename", "Ground Truth", "Prediction", "Similarity"])
    for i in range(len(y_true)):
        writer.writerow([
            selected_images[i],
            y_true[i],
            y_pred[i],
            f"{normalized_similarity(y_pred[i], y_true[i]):.4f}"
        ])

print(f"\n📁 Results saved to: {csv_path}")


Evaluation on 100 Samples:
Accuracy : 0.8571
Precision: 1.0000
Recall   : 0.8571
F1 Score : 0.9231

📁 Results saved to: tesseract_iiit5k_sample_results.csv
